In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error

# Load the training data (replace 'train.csv' with your actual file path)
train_df = pd.read_csv('train.csv', index_col='id')

# Encode 'Sex' column (1 for male, 2 for female)
train_df['Sex'] = train_df['Sex'].map({'male': 1, 'female': 2})

# Ensure numeric columns
numeric_cols = ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 'Calories']
train_df[numeric_cols] = train_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Handle missing values in training data
train_df = train_df.fillna(train_df.mean())

# Transform target to log(Calories + 1) to avoid log(0) and align with RMSLE
train_df['log_Calories'] = np.log1p(train_df['Calories'])

# Features and transformed target for training
X_train = train_df[['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp']]
y_train = train_df['log_Calories']

# Train the linear regression model on log-transformed target
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate on training data (optional, for reference)
y_train_pred_log = model.predict(X_train)
y_train_pred = np.expm1(y_train_pred_log)  # Convert back to original scale
train_rmsle = np.sqrt(mean_squared_log_error(train_df['Calories'], np.maximum(y_train_pred, 0)))
print(f"Training RMSLE: {train_rmsle:.4f}")

# Load the test data (replace 'test.csv' with your actual file path)
test_df = pd.read_csv('test.csv', index_col='id')

# Encode 'Sex' column in test data
test_df['Sex'] = test_df['Sex'].map({'male': 1, 'female': 2})

# Ensure numeric columns in test data
test_df[numeric_cols[:-1]] = test_df[numeric_cols[:-1]].apply(pd.to_numeric, errors='coerce')

# Handle missing values in test data
test_df = test_df.fillna(test_df.mean())

# Features for prediction
X_test = test_df[['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp']]

# Make predictions (in log scale) and convert back to original scale
predictions_log = model.predict(X_test)
predictions = np.expm1(predictions_log)  # expm1 ensures positive values

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': test_df.index,
    'Calories': predictions
})

# Round 'Calories' to 3 decimal places to match sample_submission.csv
submission_df['Calories'] = submission_df['Calories'].round(3)

# Save to CSV
submission_df.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully!")

Training RMSLE: 0.1799
Submission file 'submission.csv' created successfully!


In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import cross_val_score

# Load the training data (replace 'train.csv' with your actual file path)
train_df = pd.read_csv('train.csv', index_col='id')

# Encode 'Sex' column (1 for male, 2 for female)
train_df['Sex'] = train_df['Sex'].map({'male': 1, 'female': 2})

# Ensure numeric columns
numeric_cols = ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 'Calories']
train_df[numeric_cols] = train_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Handle missing values
train_df = train_df.fillna(train_df.mean())

# Feature engineering
train_df['BMI'] = train_df['Weight'] / (train_df['Height'] / 100) ** 2
train_df['Duration_Heart_Rate'] = train_df['Duration'] * train_df['Heart_Rate']
train_df['Duration_Squared'] = train_df['Duration'] ** 2
train_df['Heart_Rate_Squared'] = train_df['Heart_Rate'] ** 2

# Transform target to log(Calories + 1) for RMSLE
train_df['log_Calories'] = np.log1p(train_df['Calories'])

# Features and target
features = ['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 
            'BMI', 'Duration_Heart_Rate', 'Duration_Squared', 'Heart_Rate_Squared']
X_train = train_df[features]
y_train = train_df['log_Calories']

# Train Random Forest model
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# Evaluate on training data
y_train_pred_log = model.predict(X_train)
y_train_pred = np.expm1(y_train_pred_log)
train_rmsle = np.sqrt(mean_squared_log_error(train_df['Calories'], np.maximum(y_train_pred, 0)))
print(f"Training RMSLE: {train_rmsle:.4f}")

# Cross-validation RMSLE
scores = cross_val_score(model, X_train, y_train, cv=5, 
                         scoring='neg_mean_squared_log_error')
cv_rmsle = np.sqrt(-scores.mean())
print(f"Cross-Validation RMSLE: {cv_rmsle:.4f}")

# Load and preprocess test data (replace 'test.csv' with your actual file path)
test_df = pd.read_csv('test.csv', index_col='id')
test_df['Sex'] = test_df['Sex'].map({'male': 1, 'female': 2})
test_df[numeric_cols[:-1]] = test_df[numeric_cols[:-1]].apply(pd.to_numeric, errors='coerce')
test_df = test_df.fillna(test_df.mean())

# Feature engineering for test data
test_df['BMI'] = test_df['Weight'] / (test_df['Height'] / 100) ** 2
test_df['Duration_Heart_Rate'] = test_df['Duration'] * test_df['Heart_Rate']
test_df['Duration_Squared'] = test_df['Duration'] ** 2
test_df['Heart_Rate_Squared'] = test_df['Heart_Rate'] ** 2

# Features for prediction
X_test = test_df[features]

# Make predictions and convert back to original scale
predictions_log = model.predict(X_test)
predictions = np.expm1(predictions_log)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': test_df.index,
    'Calories': predictions
})

# Round 'Calories' to 3 decimal places
submission_df['Calories'] = submission_df['Calories'].round(3)

# Save to CSV
submission_df.to_csv('submission_rf.csv', index=False)

print("Submission file 'submission_rf.csv' created successfully!")

Training RMSLE: 0.0683
Cross-Validation RMSLE: 0.0191
Submission file 'submission_rf.csv' created successfully!


In [ ]:
# !pip install xgboost

In [ ]:
from xgboost import XGBRegressor 

In [8]:
model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_log_error')
print(f"XGBoost Cross-Validation RMSLE: {np.sqrt(-scores.mean()):.4f}")
predictions_log = model.predict(X_test)
predictions = np.expm1(predictions_log)

XGBoost Cross-Validation RMSLE: 0.0174


In [9]:
submission_df = pd.DataFrame({'id': test_df.index, 'Calories': predictions.round(3)})
submission_df.to_csv('submission_xgb.csv', index=False)

In [ ]:
# !pip install lightgbm

In [11]:
from lightgbm import LGBMRegressor
model = LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_log_error')
print(f"LightGBM Cross-Validation RMSLE: {np.sqrt(-scores.mean()):.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.076878 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 961
[LightGBM] [Info] Number of data points in the train set: 750000, number of used features: 11
[LightGBM] [Info] Start training from score 4.141144
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025772 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 962
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 11
[LightGBM] [Info] Start training from score 4.141530
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.065344 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 965
[LightGBM] [Info] Number of data points in the train set: